# Decision Trees & Random Forests
## A Completely Different Approach to Machine Learning

### CE 315 - Junior Design

**So far we've learned:**
- Linear regression: $y = w_1x_1 + w_2x_2 + ... + b$
- Logistic regression: $P(y=1) = \sigma(w_1x_1 + w_2x_2 + ... + b)$
- All models based on **linear combinations** of features

**Today's question:**
What if relationships aren't linear? What if decision-making is more like:
- "**If** age > 30 **and** has degree **then** high salary"
- "**If** temperature > 80°F **or** humidity > 70% **then** AC needed"

**Today's models:**
- **Decision Trees**: Make predictions using if-then rules
- **Random Forests**: Combine many trees for better predictions

**Real Dataset:** Titanic survival - who lived and who died?

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import LabelEncoder

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
np.random.seed(42)

print("Libraries loaded!")

---
# Part 1: The Titanic Dataset

## April 15, 1912

The "unsinkable" RMS Titanic sank on its maiden voyage, killing 1,502 of 2,224 passengers and crew.


**Our question:** Can we predict who survived based on their characteristics?

**Features:**
- **Pclass**: Ticket class (1st, 2nd, 3rd)
- **Sex**: Male or Female
- **Age**: Age in years
- **SibSp**: Number of siblings/spouses aboard
- **Parch**: Number of parents/children aboard
- **Fare**: Ticket price
- **Embarked**: Port (C = Cherbourg, Q = Queenstown, S = Southampton)

**Target:**
- **Survived**: 0 = No, 1 = Yes

In [ ]:
# Load Titanic dataset

# Try to load from seaborn
df = sns.load_dataset('titanic')
print("Data loaded from seaborn")


# Select key columns
df = df[['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']].copy()

# Drop missing values
df = df.dropna()

print(f"\nDataset: {len(df)} passengers")
print(f"\nSurvival rate: {df['survived'].mean()*100:.1f}%")
print(f"Died: {(df['survived']==0).sum()} passengers")
print(f"Survived: {(df['survived']==1).sum()} passengers")

In [ ]:
# First look at the data
print(df)

In [ ]:
# Exploratory Data Analysis
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Survival by class
pd.crosstab(df['pclass'], df['survived']).plot(kind='bar', ax=axes[0,0], color=['red', 'green'])
axes[0,0].set_xlabel('Passenger Class')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('Survival by Class')
axes[0,0].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)
axes[0,0].legend(['Died', 'Survived'])

# Survival by sex
pd.crosstab(df['sex'], df['survived']).plot(kind='bar', ax=axes[0,1], color=['red', 'green'])
axes[0,1].set_xlabel('Sex')
axes[0,1].set_ylabel('Count')
axes[0,1].set_title('Survival by Sex')
axes[0,1].set_xticklabels(['Female', 'Male'], rotation=0)
axes[0,1].legend(['Died', 'Survived'])

# Age distribution by survival
axes[0,2].hist([df[df['survived']==0]['age'], df[df['survived']==1]['age']], 
              bins=20, label=['Died', 'Survived'], color=['red', 'green'], alpha=0.6)
axes[0,2].set_xlabel('Age')
axes[0,2].set_ylabel('Count')
axes[0,2].set_title('Age Distribution')
axes[0,2].legend()

# Fare by survival
axes[1,0].hist([df[df['survived']==0]['fare'], df[df['survived']==1]['fare']], 
              bins=30, label=['Died', 'Survived'], color=['red', 'green'], alpha=0.6)
axes[1,0].set_xlabel('Fare ($)')
axes[1,0].set_ylabel('Count')
axes[1,0].set_title('Fare Distribution')
axes[1,0].legend()
axes[1,0].set_xlim(0, 300)

# Family size
df['family_size'] = df['sibsp'] + df['parch'] + 1
pd.crosstab(df['family_size'], df['survived']).plot(kind='bar', ax=axes[1,1], color=['red', 'green'])
axes[1,1].set_xlabel('Family Size')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Survival by Family Size')
axes[1,1].legend(['Died', 'Survived'])

# Survival rate by class and sex
survival_by_class_sex = df.groupby(['pclass', 'sex'])['survived'].mean().unstack()
survival_by_class_sex.plot(kind='bar', ax=axes[1,2], color=['pink', 'lightblue'])
axes[1,2].set_xlabel('Passenger Class')
axes[1,2].set_ylabel('Survival Rate')
axes[1,2].set_title('Survival Rate by Class & Sex')
axes[1,2].set_xticklabels(['1st', '2nd', '3rd'], rotation=0)
axes[1,2].legend(['Female', 'Male'])

plt.tight_layout()
plt.show()

print("\n What observations can we make?")

---
# Part 2: How Decision Trees Work

## The Core Idea: Recursive Splitting

**Example decision process:**
```
Is passenger female?
├─ Yes → Survived! (most women survived)
└─ No → Is passenger in 1st/2nd class?
    ├─ Yes → Is age < 10?
    │   ├─ Yes → Survived (young boys survived)
    │   └─ No → Died (adult men in upper class: 50/50)
    └─ No (3rd class) → Died (poor men died)
```

## The Algorithm

1. **Start with all data at the root**
2. **Find the best split:**
   - Try every feature
   - Try every possible threshold
   - Pick the split that best separates survived/died
3. **Recursively split each branch**
4. **Stop when:**
   - All samples in a node have same class
   - Maximum depth reached
   - Too few samples to split

## How to Measure "Best Split"?

**Gini Impurity** (most common):
$$G = 1 - \sum_{i=1}^{C} p_i^2$$

where $p_i$ = proportion of class $i$

**Interpretation:**
- G = 0: Pure (all same class) ← Goal
- G = 0.5: Maximum impurity (50/50 split)

**Alternative: Entropy**
$$H = -\sum_{i=1}^{C} p_i \log_2(p_i)$$

Both work similarly in practice

In [ ]:
# Demonstrate splitting concept with an example
# Let's manually see what happens with one split on 'sex'

print("Before Split (All Passengers):")
print(f"Total: {len(df)}")
print(f"Survived: {df['survived'].sum()} ({df['survived'].mean()*100:.1f}%)")
print(f"Died: {(1-df['survived']).sum()} ({(1-df['survived'].mean())*100:.1f}%)")

# Calculate Gini impurity
p_survived = df['survived'].mean()
p_died = 1 - p_survived
gini_before = 1 - (p_survived**2 + p_died**2)
print(f"Gini Impurity: {gini_before:.3f}\n")

print("="*50)
print("\nAfter Split on 'sex':\n")

# Female branch
female = df[df['sex'] == 'female']
print("Branch 1: Female")
print(f"Total: {len(female)}")
print(f"Survived: {female['survived'].sum()} ({female['survived'].mean()*100:.1f}%)")
p_surv_f = female['survived'].mean()
gini_female = 1 - (p_surv_f**2 + (1-p_surv_f)**2)
print(f"Gini: {gini_female:.3f}\n")

# Male branch
male = df[df['sex'] == 'male']
print("Branch 2: Male")
print(f"Total: {len(male)}")
print(f"Survived: {male['survived'].sum()} ({male['survived'].mean()*100:.1f}%)")
p_surv_m = male['survived'].mean()
gini_male = 1 - (p_surv_m**2 + (1-p_surv_m)**2)
print(f"Gini: {gini_male:.3f}\n")

# Weighted average Gini after split
gini_after = (len(female)/len(df) * gini_female + 
              len(male)/len(df) * gini_male)

print("="*50)
print(f"\nGini before split: {gini_before:.3f}")
print(f"Gini after split:  {gini_after:.3f}")
print(f"Improvement:       {gini_before - gini_after:.3f} ← We want this as large as possible!")
print("\n 'Sex' is an excellent first split")

---
# Part 3: Building Our First Decision Tree

In [ ]:
# Prepare the data
# Encode categorical variables
df_encoded = df.copy()
df_encoded['sex'] = (df_encoded['sex'] == 'male').astype(int)  # 1=male, 0=female

# One-hot encode embarked for handling port
embarked_dummies = pd.get_dummies(df_encoded['embarked'], prefix='embarked')
df_encoded = pd.concat([df_encoded, embarked_dummies], axis=1)


# Select features
feature_names = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'family_size',
                'embarked_C', 'embarked_Q', 'embarked_S']

X = df_encoded[feature_names].values
y = df_encoded['survived'].values

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train)} passengers")
print(f"Test set: {len(X_test)} passengers")
print(f"\nFeatures: {feature_names}")

In [ ]:
# Build a simple decision tree
tree_simple = DecisionTreeClassifier(
    max_depth=3,  # Limit depth for visualization
    random_state=42
)

tree_simple.fit(X_train, y_train)

# Evaluate
train_acc = tree_simple.score(X_train, y_train)
test_acc = tree_simple.score(X_test, y_test)

print(f"Training Accuracy: {train_acc:.3f} ({train_acc*100:.1f}%)")
print(f"Test Accuracy: {test_acc:.3f} ({test_acc*100:.1f}%)")

In [ ]:
# Visualize the tree
plt.figure(figsize=(20, 10))
plot_tree(tree_simple, 
         feature_names=feature_names,
         class_names=['Died', 'Survived'],
         filled=True,
         fontsize=10,
         rounded=True)
plt.title('Decision Tree (max_depth=3)', fontsize=16, fontweight='bold')
plt.show()

print("\n How to Read This Tree:\n")
print("Each box shows:")
print("  • Top: The decision rule (e.g., 'sex <= 0.5' means female)")
print("  • gini: Impurity measure (lower = purer)")
print("  • samples: Number of passengers in this node")
print("  • value: [# died, # survived]")
print("  • class: Final prediction (Died or Survived)")
print("\nColor:")
print("  • Orange: Predicts 'Died'")
print("  • Blue: Predicts 'Survived'")
print("  • Darker = more confident")

In [ ]:
# Let's trace a prediction manually
print("\n Example Predictions:\n")

# Example 1: Young woman in 1st class
example1 = np.array([[1, 0, 25, 0, 0, 100, 1, 0, 0, 1]])  # pclass=1, sex=0(female), age=25, etc.
pred1 = tree_simple.predict(example1)[0]
prob1 = tree_simple.predict_proba(example1)[0]

print("Passenger 1: 1st class, female, age 25")
print(f"  Decision path: sex(female) → ... ")
print(f"  Prediction: {'SURVIVED' if pred1 == 1 else 'DIED'}")
print(f"  Confidence: {prob1[pred1]*100:.1f}%\n")

# Example 2: Adult man in 3rd class
example2 = np.array([[3, 1, 30, 0, 0, 10, 1, 0, 0, 1]])  # pclass=3, sex=1(male), age=30
pred2 = tree_simple.predict(example2)[0]
prob2 = tree_simple.predict_proba(example2)[0]

print("Passenger 2: 3rd class, male, age 30")
print(f"  Decision path: sex(male) → pclass(3rd) → ...")
print(f"  Prediction: {'SURVIVED' if pred2 == 1 else 'DIED'}")
print(f"  Confidence: {prob2[pred2]*100:.1f}%\n")

# Example 3: Young boy in 2nd class
example3 = np.array([[2, 1, 8, 1, 1, 30, 3, 0, 0, 1]])  # pclass=2, sex=1(male), age=8
pred3 = tree_simple.predict(example3)[0]
prob3 = tree_simple.predict_proba(example3)[0]

print("Passenger 3: 2nd class, male, age 8 (child)")
print(f"  Decision path: sex(male) → age(young) → ...")
print(f"  Prediction: {'SURVIVED' if pred3 == 1 else 'DIED'}")
print(f"  Confidence: {prob3[pred3]*100:.1f}%")

---
# Part 4: The Overfitting Problem

**Question:** What if we let the tree grow deep?

**Spoiler:** It will overfit

In [ ]:
# Try different tree depths
depths = [1, 2, 3, 5, 10, 20, None]  # None = unlimited
train_scores = []
test_scores = []

for depth in depths:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    
    train_scores.append(tree.score(X_train, y_train))
    test_scores.append(tree.score(X_test, y_test))
    
    depth_str = str(depth) if depth is not None else 'Unlimited'
    print(f"Depth {depth_str:10s}: Train={train_scores[-1]:.3f}, Test={test_scores[-1]:.3f}")

In [ ]:
# Plot the results
depth_labels = [str(d) if d is not None else 'Unlimited' for d in depths]

plt.figure(figsize=(12, 6))
x_pos = range(len(depths))
plt.plot(x_pos, train_scores, 'o-', linewidth=2, markersize=8, label='Training Accuracy')
plt.plot(x_pos, test_scores, 's-', linewidth=2, markersize=8, label='Test Accuracy')
plt.xticks(x_pos, depth_labels)
plt.xlabel('Max Depth', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('Decision Tree Performance vs Depth', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim(0.6, 1.05)



plt.show()

print("\n Observations:")
print("1. Training accuracy keeps increasing (eventually reaches 100%!)")
print("2. Test accuracy increases then DECREASES (overfitting!)")
print(f"3. Optimal depth: {depths[best_idx]} (Test Acc = {test_scores[best_idx]:.3f})")
print("\n  Deep trees memorize training data but fail to generalize")

In [ ]:
# Visualize an overfit tree
tree_overfit = DecisionTreeClassifier(max_depth=None, random_state=42)
tree_overfit.fit(X_train, y_train)

print(f"Unlimited depth tree:")
print(f"  Actual depth reached: {tree_overfit.get_depth()}")
print(f"  Number of leaves: {tree_overfit.get_n_leaves()}")
print(f"  Training accuracy: {tree_overfit.score(X_train, y_train):.3f}")
print(f"  Test accuracy: {tree_overfit.score(X_test, y_test):.3f}")
print("\n Too many leaves = memorizing training data!")

---
# Part 5: Random Forests - Wisdom of the Crowd

## The Problem with Single Trees

- High variance (small changes in data → very different tree)
- Can overfit easily
- Unstable

## The Solution: Random Forest

**Core Idea:** Instead of one tree, build MANY trees and vote

**How it works:**
1. **Bootstrap Sampling**: Create many random subsets of data (sampling with replacement)
2. **Random Feature Selection**: At each split, only consider a random subset of features
3. **Build Trees**: Train a tree on each bootstrap sample with random features
4. **Vote**: For classification, majority vote. For regression, average.

**Why this works:**
- Each tree is slightly different (trained on different data, using different features)
- Errors of individual trees cancel out
- **Ensemble** is more robust and accurate than any single tree

**The Math:**
If you have N trees with accuracy 70% that make independent errors:
- Majority vote accuracy: ~90%
- The more trees, the better (up to a point)

In [ ]:
# Build a Random Forest
rf = RandomForestClassifier(
    n_estimators=100,  # Number of trees
    max_depth=10,      # Max depth of each tree
    random_state=42
)

rf.fit(X_train, y_train)

# Evaluate
train_acc_rf = rf.score(X_train, y_train)
test_acc_rf = rf.score(X_test, y_test)

print("Random Forest (100 trees):")
print(f"  Training Accuracy: {train_acc_rf:.3f} ({train_acc_rf*100:.1f}%)")
print(f"  Test Accuracy: {test_acc_rf:.3f} ({test_acc_rf*100:.1f}%)")

# Compare to single tree
tree_single = DecisionTreeClassifier(max_depth=10, random_state=42)
tree_single.fit(X_train, y_train)
test_acc_tree = tree_single.score(X_test, y_test)

print(f"\nSingle Tree (depth=10):")
print(f"  Test Accuracy: {test_acc_tree:.3f} ({test_acc_tree*100:.1f}%)")

print(f"\nImprovement: {(test_acc_rf - test_acc_tree)*100:.1f} percentage points")
print("\n Random Forest is more accurate and robust!")

In [ ]:
# Show how accuracy improves with more trees
n_trees_list = [1, 5, 10, 25, 50, 100, 200, 500]
rf_test_scores = []

for n_trees in n_trees_list:
    rf_temp = RandomForestClassifier(n_estimators=n_trees, max_depth=10, random_state=42)
    rf_temp.fit(X_train, y_train)
    rf_test_scores.append(rf_temp.score(X_test, y_test))
    print(f"Trees: {n_trees:3d} → Test Accuracy: {rf_test_scores[-1]:.3f}")

# Plot
plt.figure(figsize=(12, 6))
plt.plot(n_trees_list, rf_test_scores, 'o-', linewidth=2, markersize=8, color='green')
plt.axhline(test_acc_tree, color='red', linestyle='--', linewidth=2, label=f'Single Tree: {test_acc_tree:.3f}')
plt.xlabel('Number of Trees', fontsize=12)
plt.ylabel('Test Accuracy', fontsize=12)
plt.title('Random Forest: More Trees = Better Performance', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xscale('log')
plt.show()

print("\n Observations:")
print("1. Performance improves rapidly at first (1 → 50 trees)")
print("2. Diminishing returns after ~100 trees")
print("3. Always better than a single tree")
print("\n Typical choice: 100-500 trees")

---
# Part 6: Feature Importance

**Question:** Which features matter most for survival?

**Random Forests tell us**

Feature importance = how much that feature decreases impurity (averaged across all trees)

In [ ]:
# Get feature importances
importances = rf.feature_importances_
indices = np.argsort(importances)[::-1]  # Sort descending

# Print ranking
print("Feature Importance Ranking:\n")
for i, idx in enumerate(indices):
    print(f"{i+1:2d}. {feature_names[idx]:15s} {importances[idx]:.4f}")

# Plot
plt.figure(figsize=(12, 6))
colors = ['green' if importances[i] > 0.1 else 'lightblue' for i in indices]
plt.barh(range(len(importances)), importances[indices], color=colors, edgecolor='black')
plt.yticks(range(len(importances)), [feature_names[i] for i in indices])
plt.xlabel('Importance', fontsize=12)
plt.title('Feature Importance (Random Forest)', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print("\n Interpretation:")
print("Top features align with historical facts:")
print("  • Sex: 'Women and children first' policy")
print("  • Fare/Class: Wealth = access to lifeboats")
print("  • Age: Children prioritized")
print("\n Random Forests give us interpretable insights")

---
# Part 7: Comparison to Logistic Regression

Let's compare Random Forest to our old friend, Logistic Regression!

In [ ]:
# Scale features for logistic regression
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()  #why is scaling unimportant with decision trees?
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train logistic regression
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

# Evaluate both models
test_acc_lr = log_reg.score(X_test_scaled, y_test)

print("Model Comparison on Test Set:\n")
print(f"Logistic Regression: {test_acc_lr:.3f} ({test_acc_lr*100:.1f}%)")
print(f"Random Forest:       {test_acc_rf:.3f} ({test_acc_rf*100:.1f}%)")
print(f"\nWinner: {'Random Forest' if test_acc_rf > test_acc_lr else 'Logistic Regression'}")
print(f"Margin: {abs(test_acc_rf - test_acc_lr)*100:.1f} percentage points")

In [ ]:
# Detailed comparison with confusion matrices
from sklearn.metrics import confusion_matrix

# Predictions
y_pred_lr = log_reg.predict(X_test_scaled)
y_pred_rf = rf.predict(X_test)

# Confusion matrices
cm_lr = confusion_matrix(y_test, y_pred_lr)
cm_rf = confusion_matrix(y_test, y_pred_rf)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
           xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)
axes[0].set_title(f'Logistic Regression\nAccuracy: {test_acc_lr:.3f}', fontsize=13)

# Random Forest
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', cbar=False, ax=axes[1],
           xticklabels=['Died', 'Survived'], yticklabels=['Died', 'Survived'])
axes[1].set_xlabel('Predicted', fontsize=12)
axes[1].set_ylabel('Actual', fontsize=12)
axes[1].set_title(f'Random Forest\nAccuracy: {test_acc_rf:.3f}', fontsize=13)

plt.tight_layout()
plt.show()

# Classification reports
print("\nLogistic Regression:")
print(classification_report(y_test, y_pred_lr, target_names=['Died', 'Survived']))

print("\nRandom Forest:")
print(classification_report(y_test, y_pred_rf, target_names=['Died', 'Survived']))

---
# Part 8: When to Use What?

## Decision Trees

**Pros:**
- Easy to understand and visualize
-  No feature scaling needed
-  Handles non-linear relationships naturally
-  Handles mixed data types (categorical + numerical)
-  Fast to train and predict

**Cons:**
-  High variance (unstable)
-  Easy to overfit


**Best for:**
- Exploratory analysis
- When you need interpretability
- When you have categorical features

---

## Random Forests

**Pros:**
-  More accurate than single trees
-  Robust (low variance)
-  Less prone to overfitting
-  Feature importance built-in
-  Works well "out of the box"
-  Handles non-linear relationships
-  No feature scaling needed

**Cons:**
-  Less interpretable (can't visualize easily)
-  Slower to train (many trees)
-  Larger model size (memory)
-  Not great for extrapolation

**Best for:**
- When accuracy matters most
- Tabular data
- When you have mixed feature types
- Real-world production systems

---

## Logistic Regression

**Pros:**
-  Very interpretable (coefficients)
-  Probabilistic predictions
-  Fast training and prediction
-  Works well with linear relationships
-  Small model size

**Cons:**
-  Assumes linear decision boundaries
-  Requires feature scaling
-  Limited with non-linear patterns

**Best for:**
- When interpretability is critical
- When relationships are approximately linear
- When you need probability estimates
- Baseline model

---

## Rule of Thumb

**Start with:**
1. Logistic Regression (baseline)
2. Random Forest (often wins)
3. Compare using cross-validation

**If you need interpretability:**
- Use Decision Tree (single, shallow)
- Or Logistic Regression with feature selection

**If you need maximum accuracy:**
- Try Random Forest
- Try Gradient Boosting (XGBoost, LightGBM - beyond this course)
- Ensemble multiple models

---
# Part 9: Hyperparameter Tuning

Random Forests have several hyperparameters to tune:
- `n_estimators`: Number of trees
- `max_depth`: Maximum depth of each tree
- `min_samples_split`: Minimum samples to split a node
- `min_samples_leaf`: Minimum samples in a leaf
- `max_features`: Number of features to consider at each split

Let's use GridSearchCV to find the best

In [ ]:
# Define parameter grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

print("Parameter Grid:")
for param, values in param_grid.items():
    print(f"  {param}: {values}")

total_combinations = np.prod([len(v) for v in param_grid.values()])
print(f"\nTotal combinations: {total_combinations}")
print(f"With 5-fold CV: {total_combinations * 5} model fits!")
print("\nThis may take a minute...")

In [ ]:
# Run GridSearchCV
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1  # Use all CPU cores
)

grid_search.fit(X_train, y_train)

print("\n GridSearchCV Complete\n")
print("Best Parameters:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

print(f"\nBest CV Score: {grid_search.best_score_:.3f}")

# Test on hold-out set
best_rf = grid_search.best_estimator_
test_acc_best = best_rf.score(X_test, y_test)
print(f"Test Accuracy: {test_acc_best:.3f}")

